In [1]:
import pandas as pd

In [2]:
ratings = pd.read_csv("../data/raw/ratings.csv")
movies = pd.read_csv("../data/raw/movies.csv")

ratings_small = ratings.sample(200000, random_state=42)

ratings_small.head()

,userId,movieId,rating,timestamp
15347762,99476,104374,3.5,1467897440
16647840,107979,2634,4.0,994007728
23915192,155372,1614,3.0,1097887531
10052313,65225,7153,4.0,1201382275
12214125,79161,500,5.0,1488915363


In [3]:
!pip install scikit-surprise

In [4]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

In [5]:
reader = Reader(rating_scale=(0.5, 5.0))

data = Dataset.load_from_df(
    ratings_small[["userId", "movieId", "rating"]],
    reader
)

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

svd_model = SVD()
svd_model.fit(trainset)

predictions = svd_model.test(testset)

accuracy.rmse(predictions)

RMSE: 0.9541


0.9541391122997008

In [6]:
def recommend_svd(user_id, top_n=10):
    all_movie_ids = movies["movieId"].unique()

    watched_movies = ratings_small[
        ratings_small["userId"] == user_id
    ]["movieId"].unique()

    unwatched_movies = [
        movie_id for movie_id in all_movie_ids
        if movie_id not in watched_movies
    ]

    predicted_ratings = []

    for movie_id in unwatched_movies[:10000]:
        pred = svd_model.predict(user_id, movie_id)
        predicted_ratings.append((movie_id, pred.est))

    predicted_ratings = sorted(
        predicted_ratings,
        key=lambda x: x[1],
        reverse=True
    )

    top_movies = predicted_ratings[:top_n]

    result = pd.DataFrame(
        top_movies,
        columns=["movieId", "predicted_rating"]
    )

    result = result.merge(movies, on="movieId")

    return result[["movieId", "title", "genres", "predicted_rating"]]

In [7]:
recommend_svd(user_id=1, top_n=10)

,movieId,title,genres,predicted_rating
0,318,"Shawshank Redemption, The (1994)",Crime|Drama,4.516819
1,922,Sunset Blvd. (a.k.a. Sunset Boulevard) (1950),Drama|Film-Noir|Romance,4.502599
2,1234,"Sting, The (1973)",Comedy|Crime,4.490812
3,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.480399
4,1213,Goodfellas (1990),Crime|Drama,4.450165
5,3462,Modern Times (1936),Comedy|Drama|Romance,4.444865
6,908,North by Northwest (1959),Action|Adventure|Mystery|Romance|Thriller,4.426382
7,1178,Paths of Glory (1957),Drama|War,4.413703
8,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,4.399867
9,1197,"Princess Bride, The (1987)",Action|Adventure|Comedy|Fantasy|Romance,4.383588
